# Lab — Building Blocks, Measured

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s4-building-blocks.ipynb)

This notebook takes you from nothing to a submission on challenge 8, *1 Minute
Permuted MNIST*: the problem, a baseline agent (logistic regression in PyTorch), the
challenge's evaluation run locally (accuracy and timing), and the submission. The
last section is yours: what to change in the agent to raise its score, and how to
test each change.

---

## 1. The problem

Your agent is a Python class. The challenge builds it once, calls
`train(X_train, y_train)`, then `predict(X_test)`, and scores the accuracy of the
predictions.

| | |
|---|---|
| `X_train` | uint8 `(60000, 28, 28)`: MNIST images with the **pixel positions permuted** |
| `y_train` | int64 `(60000, 1)`: labels 0–9 with the **label meanings permuted** |
| `X_test` | uint8 `(10000, 28, 28)`, same permutation |
| `predict` returns | a list of 10,000 ints |

- `train` and `predict` each have a **60-second deadline**, on **3 CPU cores and
  3 GiB**, no GPU. A missed deadline or an exception scores 0.
- The pixel permutation removes the spatial structure: the model sees 784 unordered
  inputs, so a convolution has nothing to exploit.
- Every task draws a new permutation. If the accuracy reaches 0.98, the challenge
  trains the same agent again on a permuted **Fashion-MNIST** task and expects at
  least 0.40 there; an agent that fails this check scores −1.
- The upload validator checks the method signatures:
  `__init__(self, output_dim: int = 10, seed=None)`, `train(self, X_train, y_train)`,
  `predict(self, X_test)`.

The objective is a trade-off: the best accuracy a model can reach with less than 60
seconds of CPU training.

In [ ]:
import importlib
import importlib.util
import os
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
if importlib.util.find_spec("mlarena") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlarena-sdk"], check=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision

torch.set_num_threads(3)          # the grading container has 3 cores: measure like it
print(f"torch {torch.__version__}, threads {torch.get_num_threads()}")

`make_task` applies to torchvision's MNIST what the challenge applies: one random
permutation of the 784 pixel positions, one of the 10 labels, and mild noise.

In [ ]:
def make_task(images_train, labels_train, images_test, labels_test, seed):
    """Permute the pixel positions and the label meanings, add the challenge's mild noise."""
    rng = np.random.RandomState(seed)
    label_perm, pixel_perm = rng.permutation(10), rng.permutation(28 * 28)

    def permute(images):
        flat = images.reshape(len(images), -1)[:, pixel_perm].astype(np.float32) / 255.0
        flat += rng.normal(0, 0.015, flat.shape).astype(np.float32)
        flat = flat * rng.uniform(0.96, 1.04, (len(flat), 1)).astype(np.float32)
        flat += rng.uniform(-0.02, 0.02, (len(flat), 1)).astype(np.float32)
        return (np.clip(flat, 0, 1) * 255).astype(np.uint8).reshape(-1, 28, 28)

    return {"X_train": permute(images_train), "y_train": label_perm[labels_train].reshape(-1, 1).astype(np.int64),
            "X_test": permute(images_test), "y_test": label_perm[labels_test].astype(np.int64)}


mnist_train = torchvision.datasets.MNIST("data", train=True, download=True)
mnist_test = torchvision.datasets.MNIST("data", train=False, download=True)
task = make_task(mnist_train.data.numpy(), mnist_train.targets.numpy(),
                 mnist_test.data.numpy(), mnist_test.targets.numpy(), seed=1)
print({k: (v.shape, v.dtype) for k, v in task.items()})

fig, axes = plt.subplots(1, 2, figsize=(5, 2.6))
axes[0].imshow(mnist_train.data[0], cmap="gray"); axes[0].set_title(f"MNIST: label {int(mnist_train.targets[0])}")
axes[1].imshow(task["X_train"][0], cmap="gray"); axes[1].set_title(f"permuted: label {int(task['y_train'][0, 0])}")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

---

## 2. Baseline: logistic regression in PyTorch

Logistic regression is one linear layer, 784 inputs to 10 logits, trained with
cross-entropy (the softmax is inside the loss). The cell writes the agent to
`agent.py`, the file you submit. Four things in it are rules; keep them in every
version:

- a **new model on every `train()` call**: the Fashion-MNIST check calls it a second time;
- normalization statistics computed from the `X_train` that arrives, never hard-coded;
- a **wall-clock guard**: training stops at `TRAIN_BUDGET_S`, well before 60 s,
  because the grading machine can be slower than yours;
- `model.eval()` and `torch.no_grad()` in `predict`.

In [ ]:
%%writefile agent.py
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

TRAIN_BUDGET_S = 40.0     # of the 60 s deadline: leave a margin for a slower machine
EPOCHS = 5
BATCH = 256
LR = 1e-3


class Agent:
    def __init__(self, output_dim: int = 10, seed=None):
        torch.set_num_threads(3)
        self.output_dim = output_dim
        self.seed = 0 if seed is None else seed

    @staticmethod
    def _flatten(X):
        return torch.from_numpy(np.asarray(X).reshape(len(X), -1).astype(np.float32) / 255.0)

    def train(self, X_train, y_train):
        t0 = time.perf_counter()
        torch.manual_seed(self.seed)
        X = self._flatten(X_train)
        self.mu, self.sd = X.mean(), X.std()               # statistics of THIS task
        X = (X - self.mu) / self.sd
        y = torch.from_numpy(np.asarray(y_train).reshape(-1).astype(np.int64))

        self.model = nn.Linear(X.shape[1], self.output_dim)   # a new model on every call
        opt = torch.optim.Adam(self.model.parameters(), lr=LR)
        self.model.train()
        for epoch in range(EPOCHS):
            for idx in torch.randperm(len(X)).split(BATCH):
                loss = F.cross_entropy(self.model(X[idx]), y[idx])
                opt.zero_grad()
                loss.backward()
                opt.step()
                if time.perf_counter() - t0 > TRAIN_BUDGET_S:  # the wall-clock guard
                    return

    def predict(self, X_test):
        self.model.eval()
        with torch.no_grad():
            return self.model((self._flatten(X_test) - self.mu) / self.sd).argmax(1).tolist()

---

## 3. Evaluation: score and timing

`evaluate_agent` does what the challenge does: it builds the agent from `agent.py`,
times `train` and `predict` against the 60 s deadlines, scores the predictions, and
runs the Fashion-MNIST check when the accuracy reaches 0.98. Every call adds a row
to the table, so the table keeps the history of what you tried. Re-run this cell
after every edit of `agent.py`, with a new name.

In [ ]:
import agent

RESULTS = []


def evaluate_agent(name, deadline_s=60.0):
    importlib.reload(agent)                              # pick up the last edit of agent.py
    bot = agent.Agent()
    t0 = time.perf_counter(); bot.train(task["X_train"], task["y_train"]); train_s = time.perf_counter() - t0
    t0 = time.perf_counter(); pred = np.asarray(bot.predict(task["X_test"])).ravel(); predict_s = time.perf_counter() - t0
    row = {"name": name, "accuracy": float((pred == task["y_test"]).mean()),
           "train_s": round(train_s, 1), "predict_s": round(predict_s, 1),
           "within_deadlines": train_s < deadline_s and predict_s < deadline_s}
    if row["accuracy"] >= 0.98:                          # the challenge's check: same agent, another dataset
        f_train = torchvision.datasets.FashionMNIST("data", train=True, download=True)
        f_test = torchvision.datasets.FashionMNIST("data", train=False, download=True)
        fashion = make_task(f_train.data.numpy(), f_train.targets.numpy(),
                            f_test.data.numpy(), f_test.targets.numpy(), seed=2)
        bot.train(fashion["X_train"], fashion["y_train"])
        row["fashion_accuracy"] = float((np.asarray(bot.predict(fashion["X_test"])).ravel() == fashion["y_test"]).mean())
    RESULTS.append(row)
    return pd.DataFrame(RESULTS)


evaluate_agent("logistic regression")

---

## 4. Submit

The key comes from Colab's *Secrets* panel as `MLARENA_API_KEY` (ML-Arena, Profile →
API Keys, it starts with `mlk_user_`), granted to this notebook; outside Colab, from
the environment. Never paste it in a cell. `wait=True` returns once the evaluation
has settled, which takes a few minutes.

In [ ]:
import mlarena

if IN_COLAB:
    from google.colab import userdata
    MLARENA_API_KEY = userdata.get("MLARENA_API_KEY")
else:
    MLARENA_API_KEY = os.environ["MLARENA_API_KEY"]

client = mlarena.connect(api_key=MLARENA_API_KEY)
submission = client.submit(8, files=["agent.py"], submission_name="logistic-regression",
                           runtime={"language": "python", "framework": "torch"}, wait=True)
print(submission["status"]["status"], "--", submission["status"]["last_status_message"])
print(client.leaderboard(8, top=10))

---

## 5. Improve the score

The score has two sides: accuracy, and 60 seconds of training on 3 cores. Every idea
below trades one against the other, so test each one the same way: **change one
thing in `agent.py`, re-run section 3 with a new name, and compare the new row with
the previous one**, accuracy *and* `train_s`. Keep a change only if it raises the
accuracy with `train_s` still well under 60. Then submit again with a new
`submission_name`.

| Lever | What to try | What to watch |
|---|---|---|
| Architecture | hidden layers with ReLU: `784 → 256 → 10`, then `784 → 256 → 256 → 10`; wider vs deeper | the largest gain; the time per epoch grows with the width |
| Optimizer | Adam, AdamW, SGD with momentum 0.9; the learning rate ×3 and ÷3 | accuracy at equal training time |
| Schedule | cosine or OneCycle, to end training on a small learning rate | the last epochs' accuracy |
| Batch size | 128, 256, 512 | throughput on 3 cores against the number of updates |
| Normalization | `BatchNorm1d` or `LayerNorm` after each hidden `Linear` | a larger learning rate becomes stable; `model.eval()` becomes mandatory |
| Dropout | `Dropout(0.1)` to `Dropout(0.3)` after each activation | helps only if the model overfits |
| Data augmentation | Gaussian noise on the inputs, randomly zeroed pixels | rotations and shifts mean nothing once the pixels are permuted |
| Early stopping | hold out 5–10% of `X_train`, keep the weights of the best epoch | whether the last epochs still help |

Three things decide which of these are worth your time:

- **Measure the gap before you regularize.** Hold out part of `X_train` inside
  `train()` and print the training and validation accuracy of each epoch. Within 40
  seconds a small network is often still underfitting; then dropout and
  augmentation lower the accuracy, and a larger model or a better schedule raises it.
- **Time is a hyperparameter.** Time one epoch, then run as many as
  `TRAIN_BUDGET_S` allows, rather than a fixed `EPOCHS`. Keep the margin: the grading
  machine can be slower than Colab.
- **Past 0.98, the Fashion-MNIST check runs.** Check that `fashion_accuracy` is at
  least 0.40 in section 3 before you submit.

For reference, logistic regression scores about 0.92; a two-hidden-layer ReLU MLP
with BatchNorm, AdamW and an annealed learning rate reaches about 0.987 in 37
seconds on a laptop.